# MLOps Training — Task 2
## Notebook 3 — Data Splitting Strategy & Leakage Prevention

### Objective
In this notebook, we split the labeled dataset chronologically into three isolated subsets:
- **Train Set (70%)**: Earlier orders, used for fitting preprocessing transformations and training machine learning models.
- **Validation Set (15%)**: Later orders, used for hyperparameter tuning and decision threshold optimization.
- **Test Set (15%)**: The latest orders, reserved for final unbiased performance evaluation.

We also verify strict partition isolation and temporal ordering so that no future orders enter model-development data.

### Input Artifact
- `artifacts/notebook_02/olist_labeled_table.parquet`

### Output Artifacts
- `artifacts/notebook_03/train.parquet`
- `artifacts/notebook_03/validation.parquet`
- `artifacts/notebook_03/test.parquet`


## 1. Import Libraries & Configure Paths

In [1]:
from pathlib import Path

import pandas as pd

# Reproducible split configuration
TARGET = "is_late"
ORDER_ID_COLUMN = "order_id"
TEMPORAL_SPLIT_COLUMN = "order_purchase_timestamp"
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

# Dynamic project paths
artifact_dir = Path.cwd() / "artifacts" / "notebook_03"
artifact_dir.mkdir(parents=True, exist_ok=True)

input_path = Path.cwd() / "artifacts" / "notebook_02" / "olist_labeled_table.parquet"
print("Input artifact path:", input_path.resolve())
print("Output artifact dir:", artifact_dir.resolve())


Input artifact path: /Users/ouahibaahmid/training mlops/artifacts/notebook_02/olist_labeled_table.parquet
Output artifact dir: /Users/ouahibaahmid/training mlops/artifacts/notebook_03


## 2. Load Labeled Table Artifact

In [2]:
labeled_table = pd.read_parquet(input_path)

required_columns = [TARGET, ORDER_ID_COLUMN, TEMPORAL_SPLIT_COLUMN]
missing_columns = [col for col in required_columns if col not in labeled_table.columns]
if missing_columns:
    raise ValueError(f"Input artifact is missing required columns: {missing_columns}")

print(f"✓ Loaded: {input_path.name}")
print(f"Total Rows: {len(labeled_table):,}")
print(f"Total Columns: {len(labeled_table.columns)}")
print("\nTarget distribution:")
print(labeled_table[TARGET].value_counts(normalize=True).sort_index().round(4) * 100)


✓ Loaded: olist_labeled_table.parquet
Total Rows: 96,476
Total Columns: 28

Target distribution:
is_late
0    91.89
1     8.11
Name: proportion, dtype: float64


## 3. Ensure Proper Datetime Types & Analyze the Order Timeline

We parse timestamp columns before splitting. The temporal split uses **`order_purchase_timestamp`**, because it marks when an order enters the system and is available when a delivery-risk prediction would be made. The date-range analysis below confirms the available historical timeline before the split is created.


In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    if col in labeled_table.columns:
        labeled_table[col] = pd.to_datetime(labeled_table[col], errors="coerce")

missing_split_dates = labeled_table[TEMPORAL_SPLIT_COLUMN].isna().sum()
if missing_split_dates:
    raise ValueError(
        f"{TEMPORAL_SPLIT_COLUMN} has {missing_split_dates:,} missing values; "
        "a leakage-safe temporal split requires a valid order date for every row."
    )

if labeled_table[ORDER_ID_COLUMN].isna().any():
    raise ValueError(f"{ORDER_ID_COLUMN} contains missing values.")

if labeled_table[ORDER_ID_COLUMN].duplicated().any():
    raise ValueError(
        "The labeled table contains more than one row per order. "
        "Aggregate to one row per order before creating order-level splits."
    )

date_range_summary = pd.DataFrame(
    [
        {
            "timestamp_column": col,
            "first_timestamp": labeled_table[col].min(),
            "last_timestamp": labeled_table[col].max(),
            "missing_values": labeled_table[col].isna().sum()
        }
        for col in date_columns
        if col in labeled_table.columns
    ]
)

print("Timestamp summary:")
print(date_range_summary)

monthly_timeline = (
    labeled_table
    .assign(order_month=labeled_table[TEMPORAL_SPLIT_COLUMN].dt.to_period("M").astype(str))
    .groupby("order_month", as_index=False)
    .agg(
        orders=(ORDER_ID_COLUMN, "size"),
        late_rate_pct=(TARGET, lambda values: values.mean() * 100)
    )
)
monthly_timeline["late_rate_pct"] = monthly_timeline["late_rate_pct"].round(2)

print("\nMonthly order timeline used for the chronological split:")
print(monthly_timeline)


Timestamp summary:
                timestamp_column     first_timestamp      last_timestamp  \
0       order_purchase_timestamp 2016-09-15 12:16:38 2018-08-29 15:00:37   
1  order_delivered_customer_date 2016-10-11 13:46:32 2018-10-17 13:22:46   
2  order_estimated_delivery_date 2016-10-04 00:00:00 2018-10-25 00:00:00   

   missing_values  
0               0  
1               0  
2               0  

Monthly order timeline used for the chronological split:
   order_month  orders  late_rate_pct
0      2016-09       1         100.00
1      2016-10     270           1.11
2      2016-12       1           0.00
3      2017-01     750           3.07
4      2017-02    1653           3.21
5      2017-03    2546           5.58
6      2017-04    2303           7.86
7      2017-05    3545           3.61
8      2017-06    3135           3.86
9      2017-07    3872           3.43
10     2017-08    4193           3.32
11     2017-09    4150           5.20
12     2017-10    4478           5.29
13    

## 4. Splitting Strategy: Temporal Out-of-Time Split

> **MLOps Methodology Note**:
> A production delivery-risk model is trained using orders already observed, then asked to predict newly arriving orders. A random split mixes earlier and later orders in every partition, allowing development data to contain future operating conditions such as changing demand, seasonality, and logistics performance.
>
> Therefore, this notebook uses a **temporal (out-of-time) split** based on `order_purchase_timestamp`:
>
> ```text
> Earlier orders  →  Train (70%)
> Later orders    →  Validation (15%)
> Latest orders   →  Test (15%)
> ```
>
> We sort by purchase timestamp and use `order_id` only as a deterministic tie-breaker. Split boundaries are moved to the next timestamp when needed so the same purchase timestamp never appears in two partitions. We intentionally do **not** stratify: label balance is checked after splitting, but preserving real time-based label drift is more realistic than forcing future orders to match the past.


In [4]:
chronological_table = (
    labeled_table
    .sort_values(
        [TEMPORAL_SPLIT_COLUMN, ORDER_ID_COLUMN],
        kind="mergesort"
    )
    .reset_index(drop=True)
)


def move_to_next_timestamp_boundary(df, boundary, date_column):
    # Keep all orders with the same purchase timestamp in one partition.
    while (
        0 < boundary < len(df)
        and df.loc[boundary - 1, date_column] == df.loc[boundary, date_column]
    ):
        boundary += 1
    return boundary


total = len(chronological_table)
requested_train_end = int(total * TRAIN_FRACTION)
requested_validation_end = int(total * (TRAIN_FRACTION + VALIDATION_FRACTION))

train_end = move_to_next_timestamp_boundary(
    chronological_table,
    requested_train_end,
    TEMPORAL_SPLIT_COLUMN
)
validation_end = move_to_next_timestamp_boundary(
    chronological_table,
    max(requested_validation_end, train_end + 1),
    TEMPORAL_SPLIT_COLUMN
)

if not 0 < train_end < validation_end < total:
    raise ValueError("Unable to create three non-empty chronological partitions.")

train_df = chronological_table.iloc[:train_end].copy().reset_index(drop=True)
validation_df = chronological_table.iloc[train_end:validation_end].copy().reset_index(drop=True)
test_df = chronological_table.iloc[validation_end:].copy().reset_index(drop=True)

assert train_df[TEMPORAL_SPLIT_COLUMN].max() < validation_df[TEMPORAL_SPLIT_COLUMN].min()
assert validation_df[TEMPORAL_SPLIT_COLUMN].max() < test_df[TEMPORAL_SPLIT_COLUMN].min()

split_sizes = pd.DataFrame(
    {
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "percentage": [
            len(train_df) / total * 100,
            len(validation_df) / total * 100,
            len(test_df) / total * 100
        ],
        "first_order_date": [
            train_df[TEMPORAL_SPLIT_COLUMN].min(),
            validation_df[TEMPORAL_SPLIT_COLUMN].min(),
            test_df[TEMPORAL_SPLIT_COLUMN].min()
        ],
        "last_order_date": [
            train_df[TEMPORAL_SPLIT_COLUMN].max(),
            validation_df[TEMPORAL_SPLIT_COLUMN].max(),
            test_df[TEMPORAL_SPLIT_COLUMN].max()
        ]
    },
    index=["Train", "Validation", "Test"]
)
split_sizes["percentage"] = split_sizes["percentage"].round(2)

print("Chronological split sizes and date ranges:")
print(split_sizes)


Chronological split sizes and date ranges:
             rows  percentage    first_order_date     last_order_date
Train       67533        70.0 2016-09-15 12:16:38 2018-04-15 20:07:56
Validation  14471        15.0 2018-04-15 20:10:23 2018-06-21 07:50:39
Test        14472        15.0 2018-06-21 08:29:29 2018-08-29 15:00:37


## 5. Check Label Balance across Temporal Splits

Unlike a random stratified split, an out-of-time split does not force each partition to have the same label ratio. We report the on-time and late-delivery balance for each period to identify any time-based class drift that the model will face in production.


In [5]:
def summarize_label_balance(df, split_name):
    counts = df[TARGET].value_counts().reindex([0, 1], fill_value=0)
    return {
        "split": split_name,
        "orders": len(df),
        "on_time_count": counts.loc[0],
        "late_count": counts.loc[1],
        "on_time_pct": counts.loc[0] / len(df) * 100,
        "late_pct": counts.loc[1] / len(df) * 100
    }


split_summary = pd.DataFrame(
    [
        summarize_label_balance(train_df, "Train"),
        summarize_label_balance(validation_df, "Validation"),
        summarize_label_balance(test_df, "Test")
    ]
).set_index("split")

split_summary[["on_time_pct", "late_pct"]] = (
    split_summary[["on_time_pct", "late_pct"]].round(2)
)

print("Label balance by chronological split:")
print(split_summary)

late_rate_range = split_summary["late_pct"].max() - split_summary["late_pct"].min()
print(f"Late-label rate range across periods: {late_rate_range:.2f} percentage points")
print(
    "Note: this variation is retained intentionally; it reflects the future data "
    "distribution that a production model may need to handle."
)


Label balance by chronological split:
            orders  on_time_count  late_count  on_time_pct  late_pct
split                                                               
Train        67533          61439        6094        90.98      9.02
Validation   14471          13700         771        94.67      5.33
Test         14472          13516         956        93.39      6.61
Late-label rate range across periods: 3.69 percentage points
Note: this variation is retained intentionally; it reflects the future data distribution that a production model may need to handle.


## 6. Zero-Leakage and Temporal Isolation Verification

We execute strict checks that:
1. No `order_id` appears in more than one partition.
2. Every original row and order is accounted for exactly once.
3. Train contains only earlier orders than Validation, and Validation contains only earlier orders than Test.


In [6]:
train_ids = set(train_df[ORDER_ID_COLUMN])
val_ids = set(validation_df[ORDER_ID_COLUMN])
test_ids = set(test_df[ORDER_ID_COLUMN])
source_ids = set(labeled_table[ORDER_ID_COLUMN])

print("Overlap checks:")
print("  Train ∩ Validation:", len(train_ids & val_ids))
print("  Train ∩ Test:      ", len(train_ids & test_ids))
print("  Validation ∩ Test: ", len(val_ids & test_ids))

assert len(train_ids & val_ids) == 0, "Leakage detected between Train and Validation!"
assert len(train_ids & test_ids) == 0, "Leakage detected between Train and Test!"
assert len(val_ids & test_ids) == 0, "Leakage detected between Validation and Test!"

total_split_rows = len(train_df) + len(validation_df) + len(test_df)
total_split_orders = len(train_ids | val_ids | test_ids)

print("\nAll-rows and all-orders checks:")
print("  Original rows:       ", len(labeled_table))
print("  Rows across splits:  ", total_split_rows)
print("  Original orders:     ", len(source_ids))
print("  Orders across splits:", total_split_orders)

assert total_split_rows == len(labeled_table), "Row count mismatch!"
assert total_split_orders == len(source_ids), "Order count mismatch!"
assert train_ids | val_ids | test_ids == source_ids, "Some orders are missing or unexpected!"

print("\nTemporal boundary checks:")
print(
    "  Train end < Validation start:",
    train_df[TEMPORAL_SPLIT_COLUMN].max() < validation_df[TEMPORAL_SPLIT_COLUMN].min()
)
print(
    "  Validation end < Test start: ",
    validation_df[TEMPORAL_SPLIT_COLUMN].max() < test_df[TEMPORAL_SPLIT_COLUMN].min()
)

assert train_df[TEMPORAL_SPLIT_COLUMN].max() < validation_df[TEMPORAL_SPLIT_COLUMN].min()
assert validation_df[TEMPORAL_SPLIT_COLUMN].max() < test_df[TEMPORAL_SPLIT_COLUMN].min()

print("✓ Zero leakage confirmed: partitions are mutually exclusive and strictly chronological.")


Overlap checks:
  Train ∩ Validation: 0
  Train ∩ Test:       0
  Validation ∩ Test:  0

All-rows and all-orders checks:
  Original rows:        96476
  Rows across splits:   96476
  Original orders:      96476
  Orders across splits: 96476

Temporal boundary checks:
  Train end < Validation start: True
  Validation end < Test start:  True
✓ Zero leakage confirmed: partitions are mutually exclusive and strictly chronological.


## 7. Save Parquet Split Artifacts & Verify Reload

In [7]:
train_path = artifact_dir / "train.parquet"
validation_path = artifact_dir / "validation.parquet"
test_path = artifact_dir / "test.parquet"

train_df.to_parquet(train_path, engine="pyarrow", index=False)
validation_df.to_parquet(validation_path, engine="pyarrow", index=False)
test_df.to_parquet(test_path, engine="pyarrow", index=False)

print(f"✓ Saved chronological Train:      {train_path.name}")
print(f"✓ Saved chronological Validation: {validation_path.name}")
print(f"✓ Saved chronological Test:       {test_path.name}")

# Reload check: downstream notebooks can continue to use the same artifact paths.
train_check = pd.read_parquet(train_path)
validation_check = pd.read_parquet(validation_path)
test_check = pd.read_parquet(test_path)

print("\nReload verification:")
print("  Train:     ", train_check.shape)
print("  Validation:", validation_check.shape)
print("  Test:      ", test_check.shape)

assert len(train_check) + len(validation_check) + len(test_check) == len(labeled_table)
assert train_check[TEMPORAL_SPLIT_COLUMN].max() < validation_check[TEMPORAL_SPLIT_COLUMN].min()
assert validation_check[TEMPORAL_SPLIT_COLUMN].max() < test_check[TEMPORAL_SPLIT_COLUMN].min()

print("✓ Reloaded artifacts preserve the chronological partition boundaries.")


✓ Saved chronological Train:      train.parquet
✓ Saved chronological Validation: validation.parquet
✓ Saved chronological Test:       test.parquet

Reload verification:
  Train:      (67533, 28)
  Validation: (14471, 28)
  Test:       (14472, 28)
✓ Reloaded artifacts preserve the chronological partition boundaries.


## 8. Summary & Key Takeaways

1. **Split Dimensions**: The dataset is divided into approximately 70% Train, 15% Validation, and 15% Test, with the newest orders reserved for Test.
2. **Production-realistic evaluation**: Training and tuning use only historical orders; the final test set represents later, unseen orders that more closely resemble future production predictions.
3. **Label balance monitored**: The on-time and late-delivery rates are reported for each time period. They are not artificially equalized, so real temporal drift remains visible.
4. **Partition and temporal isolation**: The notebook verifies zero `order_id` overlap, all rows/orders accounted for, and strictly increasing purchase-date ranges from Train to Validation to Test.
5. **Artifacts Ready**: The same Parquet paths in `artifacts/notebook_03/` are retained for Notebook 4 and Notebook 5. Re-run downstream notebooks so their derived artifacts reflect this new temporal split.
